# Train the CatBoost model

Runs the code cells of `run_modeling.ipynb` in order, unchanged except that it skips the plot cell, and saves two things
that notebook does not: the validation-match probabilities of the model trained without those matches (before
`REFIT_ALL`), and of the final model refit on all ten (after `REFIT_ALL`). `choose_decoder.ipynb` picks the decoder
setting from them.

Writes `models/possession_v1/` and `artifacts/possession_v1_val_probs.npz` (about 30 minutes on a 16-thread CPU). If
both already exist it does nothing: delete them to train from scratch. Run `download_external.ipynb` first.

In [ ]:
import os
from pathlib import Path

PIPELINE_DIR = Path.cwd().resolve()     # notebooks run from possession/
assert (PIPELINE_DIR / "run_decoder.ipynb").is_file(), "run this notebook from the possession/ folder"
REPO_ROOT = PIPELINE_DIR.parent
import json
import sys
import time

import numpy as np

NAME = "possession_v1"
BUNDLE = PIPELINE_DIR / "models" / NAME / "bundle.json"
VAL_OUT = PIPELINE_DIR / "artifacts" / (NAME + "_val_probs.npz")


def save_val(ns, prefix, store):
    clf, temp = ns["clf_ext"], ns["temp_ext"]
    for m, v in ns["ext"].val.items():
        store[prefix + "_probs_" + m] = ns["softmax"](clf.predict_logits(v.X) / temp)
        store["y_" + m] = v.y
        store["blocks_" + m] = np.asarray(v.blocks)


def train():
    if BUNDLE.is_file() and VAL_OUT.is_file():
        print("model already trained:", BUNDLE)
        return
    sys.path.insert(0, str(PIPELINE_DIR / "_vendor"))        # kloppy, if installed apart (pip --target possession/_vendor)
    os.environ.setdefault("MPLBACKEND", "Agg")
    nb = json.loads((PIPELINE_DIR / "run_modeling.ipynb").read_text(encoding="utf-8"))
    ns, store = {"__name__": "__main__"}, {}
    start = time.time()
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] != "code":
            continue
        src = "".join(cell["source"])
        if "plot_prediction_summary(" in src:
            continue
        exec(compile(src, "run_modeling.ipynb::cell{}".format(i), "exec"), ns)
        if "PossessionClassifier().fit(ext.X_train" in src:
            save_val(ns, "heldout", store)
        if "if REFIT_ALL" in src:
            save_val(ns, "refit", store)
        print("[cell {} done, {:.0f} min]".format(i, (time.time() - start) / 60), flush=True)
    VAL_OUT.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(VAL_OUT, **store)
    print("saved", VAL_OUT)

In [ ]:
train()